# 📋 DAT09 · 미션3 보고를 저장하고 오류와 집계 검사하기 (학습자용)

---

## 실습 시간·작업 폴더·연결 파일

- **예상 실습 시간**: 2.5시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_practice`
- **연결 파일**: 첫 셀에서 `DAT09.md`를 연결하세요.

## 이전 미션 연계와 실행 결과

- **이전 미션 연계**: 같은 작업 폴더에서 미션1~미션2을 먼저 실행했다면, 첫 셀이 `src/m01.py`~`src/m02.py`를 불러와 작업 사전 `c`를 자동으로 복원합니다.

- 각 단계 함수(`m03_s1`~`m03_s5`)의 `TODO`를 직접 작성한 뒤 셀을 실행하세요.
- 실행 결과는 `outputs/M03/DAT09_M03_result.html`과 `step_01.csv`~`step_05.csv`에 저장됩니다.



In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'student'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

def hydrate_previous_missions(c, up_to=2):
    """같은 작업 폴더에 저장된 이전 미션(src/m01.py~)을 순차 실행해 작업 사전 c를 복원합니다."""
    import importlib.util
    src_dir = c['root'] / 'src'
    missing = [k for k in range(1, up_to + 1) if not (src_dir / f'm{k:02d}.py').is_file()]
    if missing:
        miss_str = ', '.join(f'미션{k}(DAT09_m{k:02d}_*.ipynb)' for k in missing)
        print(f'[안내] 아직 완료되지 않은 이전 미션이 있습니다: {miss_str}')
        print('이전 미션 노트북을 같은 폴더에서 먼저 실행한 뒤 이 셀을 다시 실행하세요.')
        return False
    for k in range(1, up_to + 1):
        if k in c['completed']:
            continue
        mod_path = src_dir / f'm{k:02d}.py'
        spec = importlib.util.spec_from_file_location(f'dat09_prev_m{k:02d}', mod_path)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)
        fns = [getattr(mod, f'm{k:02d}_s{s}') for s in range(1, 6)]
        run_mission(c, k, fns)
    print(f'이전 미션(미션1~미션{up_to}) 상태 복원 완료.')
    return True

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
    hydrate_previous_missions(c, 2)
else:
    print('DAT09.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09.md'); hydrate_previous_missions(c)를 실행하세요.")


## 미션3 보고를 저장하고 오류와 집계 검사하기

예상 2.5시간입니다.

#### 이번 미션의 목표

보고와 후보 및 승인값을 구분해 DB에 저장하고, 오류 검사와 재고 조회를 실제 SQL 결과로 검증합니다.

### 1단계 보고와 후보 및 승인값을 분리해 DB에 저장하기

#### 이 단계에서 하는 일

프로그램을 꺼도 보고와 처리 결과가 남아야 다른 담당자가 이어서 확인할 수 있습니다. 같은 자료를 다시 읽을 때 보고가 중복으로 늘거나 후보가 확정 재고에 섞이지 않도록 저장 구조를 만듭니다.

#### 시작 전 확인

미션2까지 완료한 결과를 사용합니다. 노트북의 m03_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션2의 raw_rows, normalized와 SQLite를 사용합니다. SQLite는 별도 서버 없이 한 파일에 표를 저장하는 DB입니다. 결과 파일은 outputs/M03/integration.db입니다.

#### 작업 순서

- 원보고 raw_observation, 변환 보고 normalized_observation, 검증 이슈 validation_issue, 보완 후보 candidate, 승인 마스터 approved_master를 각각 만드세요. 각 표의 목적과 연결할 식별자를 schema.sql에 남깁니다.

- row_id로 원보고를 유일하게 식별하고, 정상 관측에는 source·site_id·item_id·metric·as_of의 중복을 막는 조건을 둡니다. 보류·격리 행도 원래 보고로 추적할 수 있어야 합니다.

- 같은 원자료와 변환 결과를 두 번 넣어 보세요. 매번 적재 전후 원보고·변환 보고의 행 수를 조회합니다. 같은 보고는 갱신하거나 건너뛰며 새 보고처럼 추가하지 않습니다.

- DB 연결과 경로를 c['conn'], c['db']에 저장하고, 적재가 끝나면 commit해 파일에 반영하세요.


#### 저장할 결과

integration.db, schema.sql과 두 번 적재한 건수 표를 남깁니다. 원래 보고 전체와 변환 보고의 대응 관계를 유지합니다.

#### 직접 확인하는 방법

첫 번째와 두 번째 적재 뒤 원보고와 변환 보고가 각각 2,160행이어야 합니다. DB를 다시 열어 같은 표와 건수를 조회하세요.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M03/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M03_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 열다섯 검증 규칙과 시험 사례 구현하기

#### 이 단계에서 하는 일

오류 건수 하나만으로는 어떤 문제인지 알 수 없습니다. 한 보고가 여러 규칙을 어길 수도 있으므로 위반한 규칙별 기록을 남깁니다. 규칙을 실제로 구현했는지는 정상·위반·경계 사례로 확인합니다.

#### 시작 전 확인

미션3의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m03_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

config/rules.json의 R01~R15, data/rule_cases.json의 45개 사례와 미션2 변환 보고를 사용합니다. R01~04는 식별·코드·지표, R05~09는 수량·단위, R10~12는 날짜·버전, R13~15는 중복·미래 관측·지연 검사입니다.

#### 작업 순서

- 보고 한 건을 받아 규칙별 참·거짓을 반환하는 함수를 작성하세요. 수량이 없는지와 숫자가 잘못됐는지, 0인지 음수인지, EA 수량이 정수인지 각각 나누어 검사합니다.

- 각 위반에 row_id·규칙 번호·처리 방식·위반 내용을 남기세요. HOLD는 보류, QUARANTINE은 격리, FLAG_STALE은 지연 표시입니다. 지연만으로 보고를 삭제하지 않습니다.

- rule_cases.json을 읽어 각 사례의 target_rule 결과와 expected_target_pass를 비교하세요. 한 규칙당 정상·위반·경계의 세 사례가 있는지 확인하고 총 45개를 실행합니다.

- 검사 함수를 c['rule_fn']에, 규칙 이슈를 DB의 validation_issue와 CSV에 저장하세요. 같은 보고의 두 위반은 이슈 두 건으로 남기되 오류 보고 수는 한 건으로 셉니다.


#### 저장할 결과

validation_issues.csv, rule_tests.json과 DB의 검증 이슈를 남깁니다. 실행시간을 측정할 때는 보고 한 건의 15규칙 전체를 한 묶음으로 측정합니다.

#### 직접 확인하는 방법

45개 사례가 모두 예상과 일치하는지 확인하세요. 60분 관측 지연은 허용하고 60분을 넘으면 지연 표시가 생겨야 합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M03/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M03_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 비교 가능한 관측값과 승인된 재고를 따로 조회하기

#### 이 단계에서 하는 일

형식이 정상인 보고라도 세 출처가 다르거나 오래됐다면 하나의 재고로 확정할 수 없습니다. 세 관측이 같은 값이라는 사실과 관리관이 승인했다는 사실도 다릅니다. 조회 단계에서 두 종류를 분리합니다.

#### 시작 전 확인

미션3의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m03_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

integration.db의 normalized_observation, approved_master와 catalog.json, expected_slots.csv를 사용합니다. 기대 목록은 비교할 전체 이력을 정할 때만 쓰고 잘못된 원코드의 자동 수정에는 쓰지 않습니다. 같은 재고 이력은 거점·품목·지표·기준시각의 조합으로 묶습니다.

#### 작업 순서

- 각 조합에 세 출처가 모두 있고, 모두 ACCEPTED이며 제때 관측했고, EA 수량이 같을 때만 consistent_observation 조회에 포함하세요. 세 수량을 더하지 않고 동일한 한 수량을 사용합니다.

- 기대하는 모든 조합에 대해 확인된 동일값·값 충돌·비교 불가 중 하나를 붙이세요. 수량이 다르면 값 충돌, 결측·격리·지연 등으로 셋을 비교할 수 없으면 비교 불가입니다.

- 승인된 재고 조회는 approved_master만 읽게 하세요. 관측이 같더라도 승인되지 않았다면 승인 재고에는 넣지 않습니다.

- 원자료의 row_id 목록과 제외 이유를 함께 남겨, 조회에서 빠진 조합을 거꾸로 찾아볼 수 있게 하세요. 묶음과 조회 결과는 뒤 미션에서 사용하도록 c에 보관합니다.


#### 저장할 결과

views.sql, logical_stock.csv와 관련 SQL 조회를 저장합니다. c['groups']와 c['logical_rows']에 묶음과 결과를 남깁니다.

#### 직접 확인하는 방법

기본 자료의 이력 조합은 720개이고, 같은 최신 수량을 확인할 수 있는 조합은 655개입니다. 나머지는 수량을 0으로 만들지 말고 이유를 남깁니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M03/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M03_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 잠정 부족량과 일치율을 분모에 맞게 계산하기

#### 이 단계에서 하는 일

부족량이 0이라는 말은 재고가 확인됐고 목표를 충족한다는 뜻입니다. 수량을 모르는 경우를 0으로 표시하면 보급 필요를 잘못 판단할 수 있습니다. 여러 시각의 재고도 서로 다른 시점이므로 더하지 않습니다.

#### 시작 전 확인

미션3의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m03_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션3의 일치 관측 조회와 logical_stock.csv, catalog.json의 목표량을 사용합니다. 먼저 결과에 사용할 기준시각 하나를 고르고 표에 그 시각을 표시합니다.

#### 작업 순서

- 선택한 시각의 품목·거점별 관측 재고와 목표량을 연결하고 max(목표량-재고,0)으로 잠정 부족량을 구하세요. 비교할 재고가 없으면 수량과 부족량 모두 N/A로 남깁니다.

- 승인 재고와 관측으로 계산한 잠정 재고를 별도 항목으로 표시하세요. 승인값이 없는 기본 실행에서는 확정 재고와 확정 부족량을 N/A로 둡니다.

- 전체 720조합 중 같은 최신 수량을 확인한 비율과, 세 출처를 비교할 수 있는 675조합 안에서 수량이 같은 비율을 따로 계산하세요. 두 비율의 분모와 뜻을 표에 적습니다.


#### 저장할 결과

baseline_metrics.csv와 잠정·확정 구분, 선택 시각, 두 일치율의 분자·분모가 있는 표를 저장합니다.

#### 직접 확인하는 방법

전체 기준 일치율은 655/720, 비교 가능한 대상 안의 일치율은 655/675입니다. 재고 90·목표 80이면 부족량 0이고, 재고가 없으면 부족량 N/A입니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M03/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M03_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 손으로 구한 값과 SQL을 대조하고 검사시간 측정하기

#### 이 단계에서 하는 일

같은 코드에서 만든 두 결과만 비교하면 같은 오류를 놓칠 수 있습니다. 작은 사례를 별도 방식으로 계산해 SQL 결과와 대조합니다. 성능은 추측하지 않고 측정 범위를 정해 잽니다.

#### 시작 전 확인

미션3의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m03_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

미션3의 조회 SQL, rule_fn, rule_tests와 15규칙 전체의 실행시간 기록을 사용합니다. 손계산 표본은 코드가 구한 합계를 그대로 복사하지 않습니다.

#### 작업 순서

- 최소 다섯 개의 작은 수량 사례를 고르고 목표량·예상 재고·예상 부족량을 손으로 계산하세요. 정상·0·초과·확인 불가를 포함해 어떤 행을 합산하거나 제외했는지 적습니다.

- 같은 사례를 SQL로 조회하고 손계산값·SQL값·차이를 한 표에 놓으세요. N/A는 두 쪽에서 모두 확인 불가인지 비교하고 숫자 차이 0과 구분합니다.

- 보고마다 15규칙 전체를 실행하는 시간을 perf_counter로 측정하고 ms로 환산하세요. 중앙값·p95·최대를 남깁니다. p95는 측정값을 작은 순으로 놓았을 때 95%가 그 이하인 지점입니다.

- 환경과 표본 수, 측정 범위를 기록하고 45개 규칙 시험의 결과도 함께 확인하세요. p95 50ms는 성능 목표로 보고하되 정확성 시험을 대신하지 않습니다.


#### 저장할 결과

손계산 대조 표, tests.log, rule_timing.json과 시험 결과를 저장합니다.

#### 직접 확인하는 방법

숫자 사례의 손계산·SQL 차이가 전부 0인지 확인하세요. p95의 단위가 초가 아닌 ms인지, 보고 전체 검사시간을 규칙 한 개의 시간과 혼동하지 않았는지 확인합니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M03/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M03_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 보고와 후보 및 승인값을 분리해 DB에 저장하기
# 할 일: 원보고 raw_observation, 변환 보고 normalized_observation, 검증 이슈 validation_issue, 보완 후보 candidate, 승인 마스터 approved_master를 각각 만드세요. 각 표의 목적과 연결할 식별자를 schema.sql에 남깁니다. row_id로 원보고를 유일하게 식별하고, 정상 관측에는 source·site_id·item_id·metric·as_of의 중복을 막는 조건을 둡니다. 보류·격리 행도 원래 보고로 추적할 수 있어야 합니다. 같은 원자료와 변환 결과를 두 번 넣어 보세요. 매번 적재 전후 원보고·변환 보고의 행 수를 조회합니다. 같은 보고는 갱신하거나 건너뛰며 새 보고처럼 추가하지 않습니다. DB 연결과 경로를 c['conn'], c['db']에 저장하고, 적재가 끝나면 commit해 파일에 반영하세요.
# 확인: 첫 번째와 두 번째 적재 뒤 원보고와 변환 보고가 각각 2,160행이어야 합니다. DB를 다시 열어 같은 표와 건수를 조회하세요.
def m03_s1(c):
    """보고와 후보 및 승인값을 분리해 DB에 저장하기.

    읽을 작업값: checks, normalized, raw_rows
    저장할 작업값: conn, db
    결과 폴더: folder(c, 3)
    """
    return TODO(3, 1)

# STEP 02 열다섯 검증 규칙과 시험 사례 구현하기
# 할 일: 보고 한 건을 받아 규칙별 참·거짓을 반환하는 함수를 작성하세요. 수량이 없는지와 숫자가 잘못됐는지, 0인지 음수인지, EA 수량이 정수인지 각각 나누어 검사합니다. 각 위반에 row_id·규칙 번호·처리 방식·위반 내용을 남기세요. HOLD는 보류, QUARANTINE은 격리, FLAG_STALE은 지연 표시입니다. 지연만으로 보고를 삭제하지 않습니다. rule_cases.json을 읽어 각 사례의 target_rule 결과와 expected_target_pass를 비교하세요. 한 규칙당 정상·위반·경계의 세 사례가 있는지 확인하고 총 45개를 실행합니다. 검사 함수를 c['rule_fn']에, 규칙 이슈를 DB의 validation_issue와 CSV에 저장하세요. 같은 보고의 두 위반은 이슈 두 건으로 남기되 오류 보고 수는 한 건으로 셉니다.
# 확인: 45개 사례가 모두 예상과 일치하는지 확인하세요. 60분 관측 지연은 허용하고 60분을 넘으면 지연 표시가 생겨야 합니다.
def m03_s2(c):
    """열다섯 검증 규칙과 시험 사례 구현하기.

    읽을 작업값: catalog, checks, conn, normalized, root
    저장할 작업값: issues, rule_fn, rule_tests, rule_timings
    결과 폴더: folder(c, 3)
    """
    return TODO(3, 2)

# STEP 03 비교 가능한 관측값과 승인된 재고를 따로 조회하기
# 할 일: 각 조합에 세 출처가 모두 있고, 모두 ACCEPTED이며 제때 관측했고, EA 수량이 같을 때만 consistent_observation 조회에 포함하세요. 세 수량을 더하지 않고 동일한 한 수량을 사용합니다. 기대하는 모든 조합에 대해 확인된 동일값·값 충돌·비교 불가 중 하나를 붙이세요. 수량이 다르면 값 충돌, 결측·격리·지연 등으로 셋을 비교할 수 없으면 비교 불가입니다. 승인된 재고 조회는 approved_master만 읽게 하세요. 관측이 같더라도 승인되지 않았다면 승인 재고에는 넣지 않습니다. 원자료의 row_id 목록과 제외 이유를 함께 남겨, 조회에서 빠진 조합을 거꾸로 찾아볼 수 있게 하세요. 묶음과 조회 결과는 뒤 미션에서 사용하도록 c에 보관합니다.
# 확인: 기본 자료의 이력 조합은 720개이고, 같은 최신 수량을 확인할 수 있는 조합은 655개입니다. 나머지는 수량을 0으로 만들지 말고 이유를 남깁니다.
def m03_s3(c):
    """비교 가능한 관측값과 승인된 재고를 따로 조회하기.

    읽을 작업값: catalog, conn, normalized, root
    저장할 작업값: groups, logical_rows
    결과 폴더: folder(c, 3)
    """
    return TODO(3, 3)

# STEP 04 잠정 부족량과 일치율을 분모에 맞게 계산하기
# 할 일: 선택한 시각의 품목·거점별 관측 재고와 목표량을 연결하고 max(목표량-재고,0)으로 잠정 부족량을 구하세요. 비교할 재고가 없으면 수량과 부족량 모두 N/A로 남깁니다. 승인 재고와 관측으로 계산한 잠정 재고를 별도 항목으로 표시하세요. 승인값이 없는 기본 실행에서는 확정 재고와 확정 부족량을 N/A로 둡니다. 전체 720조합 중 같은 최신 수량을 확인한 비율과, 세 출처를 비교할 수 있는 675조합 안에서 수량이 같은 비율을 따로 계산하세요. 두 비율의 분모와 뜻을 표에 적습니다.
# 확인: 전체 기준 일치율은 655/720, 비교 가능한 대상 안의 일치율은 655/675입니다. 재고 90·목표 80이면 부족량 0이고, 재고가 없으면 부족량 N/A입니다.
def m03_s4(c):
    """잠정 부족량과 일치율을 분모에 맞게 계산하기.

    읽을 작업값: groups, logical_rows
    저장할 작업값: baseline_metrics
    결과 폴더: folder(c, 3)
    """
    return TODO(3, 4)

# STEP 05 손으로 구한 값과 SQL을 대조하고 검사시간 측정하기
# 할 일: 최소 다섯 개의 작은 수량 사례를 고르고 목표량·예상 재고·예상 부족량을 손으로 계산하세요. 정상·0·초과·확인 불가를 포함해 어떤 행을 합산하거나 제외했는지 적습니다. 같은 사례를 SQL로 조회하고 손계산값·SQL값·차이를 한 표에 놓으세요. N/A는 두 쪽에서 모두 확인 불가인지 비교하고 숫자 차이 0과 구분합니다. 보고마다 15규칙 전체를 실행하는 시간을 perf_counter로 측정하고 ms로 환산하세요. 중앙값·p95·최대를 남깁니다. p95는 측정값을 작은 순으로 놓았을 때 95%가 그 이하인 지점입니다. 환경과 표본 수, 측정 범위를 기록하고 45개 규칙 시험의 결과도 함께 확인하세요. p95 50ms는 성능 목표로 보고하되 정확성 시험을 대신하지 않습니다.
# 확인: 숫자 사례의 손계산·SQL 차이가 전부 0인지 확인하세요. p95의 단위가 초가 아닌 ms인지, 보고 전체 검사시간을 규칙 한 개의 시간과 혼동하지 않았는지 확인합니다.
def m03_s5(c):
    """손으로 구한 값과 SQL을 대조하고 검사시간 측정하기.

    읽을 작업값: checks, conn, logical_rows, normalized, rule_tests, rule_timings
    저장할 작업값: 근거 파일과 반환 표
    결과 폴더: folder(c, 3)
    """
    return TODO(3, 5)

result_html = run_mission(c, 3, [m03_s1, m03_s2, m03_s3, m03_s4, m03_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
